In [0]:
%run ./00_config

In [0]:
items = dbutils.fs.ls(READINGS_DIR)
print("intrări în readings:", len(items))
for it in items[:10]:
    print(it.name, "|", it.size)

In [0]:
import os

# parcurge READINGS_DIR și subfolderele lui și adună toate fișierele
paths = []
for root, dirs, files in os.walk(READINGS_DIR):
    for name in files:
        paths.append(os.path.join(root, name))

paths.sort()
print("fișiere găsite:", len(paths))
for p in paths:
    print(p)

In [0]:
from itertools import islice

path = paths[0]          # schimbă indexul ca să alegi alt fișier
print("fișier:", path)
print("mărime (MB):", round(os.path.getsize(path) / 1e6, 1))
print("-----")

with open(path, "r", encoding="utf-8", errors="replace") as f:
    for i, line in enumerate(islice(f, 10)):
        print(i, repr(line))

In [0]:
n_files = 6    # câte fișiere vrei să vezi
n_lines = 4    # câte linii din fiecare

# alegem fișiere răspândite uniform: primul, ..., ultimul
step = max(1, (len(paths) - 1) // (n_files - 1))
chosen = paths[::step][:n_files]

for p in chosen:
    print("=====", os.path.basename(p), f"({round(os.path.getsize(p)/1e6, 1)} MB)")
    with open(p, "r", encoding="utf-8", errors="replace") as f:
        for line in islice(f, n_lines):
            print("   ", repr(line))

In [0]:
path = paths[0]
size = os.path.getsize(path)

with open(path, "rb") as f:
    f.seek(size // 2)      # sărim la jumătatea fișierului
    f.readline()            # aruncăm linia tăiată pe jumătate
    for _ in range(5):
        print(repr(f.readline().decode("utf-8", errors="replace")))
        

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType

raw_schema = StructType([
    StructField("lclid", StringType(), True),
    StructField("stdortou", StringType(), True),
    StructField("datetime", StringType(), True),
    StructField("kwh_raw", StringType(), True),
])

In [0]:
df = (spark.read
      .option("header", True)
      .schema(raw_schema)
      .csv(paths))

df.printSchema()
display(df)
print("rânduri:", df.count())

In [0]:
d = (df
     .withColumn("kwh_trim", F.trim("kwh_raw"))
     .withColumn("kwh", F.expr("try_cast(kwh_trim AS DOUBLE)")))

print("consum ne-numeric:", d.filter(F.col("kwh").isNull() & F.col("kwh_trim").isNotNull()).count())
print("consum gol:", d.filter(F.col("kwh_raw").isNull()).count())
print("consum negativ:", d.filter(F.col("kwh") < 0).count())

display(d.filter(F.col("kwh").isNull() & F.col("kwh_trim").isNotNull())
         .groupBy("kwh_trim").count().orderBy(F.desc("count")).limit(10))

display(d.select(
    F.min("kwh").alias("min"),
    F.max("kwh").alias("max"),
    F.avg("kwh").alias("medie"),
    F.expr("percentile_approx(kwh, 0.999)").alias("p99_9")))

In [0]:
display(d.groupBy("stdortou").count())
print("contoare distincte:", d.select("lclid").distinct().count())

t = d.withColumn("ts", F.expr("try_to_timestamp(substring(datetime, 1, 19), 'yyyy-MM-dd HH:mm:ss')"))
print("timestamp-uri neparsabile:", t.filter(F.col("ts").isNull()).count())
display(t.select(F.min("ts"), F.max("ts")))
display(t.groupBy(F.minute("ts").alias("minut")).count())